In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as f
from pyspark.sql import Window

# Initialize Spark session
spark = SparkSession.builder.appName("TransactionsDataFrame").getOrCreate()

# Define the dataset
data = [
    (1, 201, 10,  "2023-03-01", "approved"),
    (2, 201, 20,  "2023-03-05", "approved"),
    (3, 201, 30,  "2023-03-10", "approved"),
    (4, 201, 40,  "2023-03-15", "approved"),
    (5, 201, 50,  "2023-03-20", "approved"),
    (6, 201, 60,  "2023-04-01", "approved"),
    (7, 201, 70,  "2023-04-05", "approved"),
    (8, 201, 80,  "2023-04-10", "approved"),
    (9, 201, 90,  "2023-04-15", "approved"),
    (10, 201, 100, "2023-04-20", "approved"),
    (11, 202, 15,  "2023-03-02", "approved"),
    (12, 202, 25,  "2023-03-07", "approved"),
    (13, 202, 35,  "2023-03-12", "approved"),
    (14, 202, 45,  "2023-03-17", "approved"),
    (15, 202, 55,  "2023-03-22", "approved"),
    (16, 202, 65,  "2023-04-03", "approved"),
    (17, 202, 75,  "2023-04-08", "approved"),
    (18, 202, 85,  "2023-04-13", "declined"),
    (19, 202, 95,  "2023-04-18", "approved"),
    (20, 202, 105, "2023-04-23", "approved")
]

# Define schema
columns = ["transaction_id", "merchant_id", "amount", "transaction_date", "status"]

# Create DataFrame
transactions_df = spark.createDataFrame(data, columns)

# Show DataFrame
transactions_df.show()


In [0]:
result_df = (
    transactions_df.withColumn(
        "month", f.date_format(f.to_date("transaction_date"), "yyyy-MM")
    )
    .withColumn(
        "status_cnt",
        f.sum(f.when(f.col("status") == "approved", 1).otherwise(0)).over(
            Window.partitionBy("merchant_id", "month")
        ),
    )
    .select(f.col("merchant_id"), f.col("month"), f.col("status_cnt"))
    .distinct()
    .filter(f.col("status_cnt") >= 5)
    .groupBy(f.col("month"))
    .agg(f.count(f.col("merchant_id")).alias("active_merchants"))
)
display(result_df)